# Phase 3 – Step 3.9: Compliance Officer Agent — Demo

**Purpose:** watch a real ReACT loop build SAR narratives step by step, and judge the results yourself.

**What you will see**
1. Stage 1: the Risk Analyst on 4 suspicious cases, with a *simulated* human approval
2. The tools the Compliance Officer can call (its "actions")
3. Stage 2: the ReACT agent writing one narrative per approved case
4. One full Thought → Action → Observation trace
5. The final narratives with their validation reports
6. The audit trail, and tokens/cost per stage

> ⚠️ Sections 1 and 3 make **real API calls**: 4 risk analyses plus roughly 3–6 calls per
> narrative. On `gpt-4o-mini` that is a few cents at most.

## 0. Setup

In [ ]:
# Phase 3 – Step 3.9: imports, data, and both agents wired to the real client
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import textwrap

import pandas as pd

from src import config
from src.compliance_officer_agent import ComplianceError, ComplianceOfficerAgent
from src.compliance_tools import TOOL_SCHEMAS
from src.foundation_sar import DataLoader, ExplainabilityLogger, count_words
from src.llm_client import get_llm_client
from src.narrative_checks import check_narrative
from src.risk_analyst_agent import RiskAnalysisError, RiskAnalystAgent

AUDIT_FILE = config.LOG_DIR / "demo_compliance_officer.jsonl"   # logs/ is git-ignored
logger = ExplainabilityLogger(AUDIT_FILE)
cases = {c.customer.customer_id: c for c in DataLoader(logger).build_all_cases()}
client = get_llm_client()
risk_agent = RiskAnalystAgent(client, logger)
compliance_agent = ComplianceOfficerAgent(client, logger)
print(f"{len(cases)} cases | model={compliance_agent.model} | "
      f"prompts: risk={risk_agent.prompt_version}, compliance={compliance_agent.prompt_version} | "
      f"max_steps={compliance_agent.max_steps}")

## 1. Stage 1 — Risk Analyst + simulated human approval  ⚠️ real API calls
We pick one customer per planted pattern (the ID prefixes are the hidden answer key, used only to
*pick* cases). The Compliance Officer runs only on cases a human **approved** at the risk gate.
Stage 04 builds that gate for real; here we simply approve every finding that is not `Low`.

In [ ]:
# Phase 3 – Step 3.9: hidden label per customer = the planted prefix(es) on their transactions
def hidden_label(case):
    return ", ".join(sorted({t.transaction_id.split("_")[0] for t in case.transactions} - {"TXN"}))


demo_ids = [next(cid for cid, c in cases.items() if hidden_label(c) == label)
            for label in ["STRUCT", "LARGE", "RAPID", "GEO"]]

risk_results, rows = {}, []
for cid in demo_ids:
    try:
        r = risk_agent.analyze_case(cases[cid])
    except RiskAnalysisError as exc:
        rows.append({"customer": cid, "hidden_label": hidden_label(cases[cid]), "error": exc.reason})
        continue
    approved = r.risk_level != "Low"            # stand-in for the Stage 04 human gate
    if approved:
        risk_results[cid] = r
    rows.append({"customer": cid, "hidden_label": hidden_label(cases[cid]),
                 "classification": r.classification, "risk_level": r.risk_level,
                 "confidence": r.confidence_score, "approved": approved,
                 "tokens": risk_agent.last_run.token_usage.get("total_tokens", 0)})
pd.DataFrame(rows)

**Why this matters for cost:** every rejected case here saves the whole Stage 2 loop below, which
is the most expensive part. That is the "two-stage cost optimization" Stage 04 has to measure.

## 2. The agent's actions: four tools
In ReACT the model can't look things up by itself; it *asks* by calling a tool, and **our code**
answers. So facts and citations come from the case data and the curated table, not from the model's
memory. `submit_narrative` is the only way to finish, and it is accepted only if every check passes.

In [ ]:
# Phase 3 – Step 3.9: the tool schemas sent with every request
for t in TOOL_SCHEMAS:
    f = t["function"]
    print(f"{f['name']}({', '.join(f['parameters']['required'])})")
    print(textwrap.fill(f["description"], width=100, initial_indent="    ",
                        subsequent_indent="    "), "\n")

## 3. Stage 2 — Compliance Officer ReACT loop  ⚠️ real API calls

In [ ]:
# Phase 3 – Step 3.9: one narrative per approved case; failures are caught, as the workflow will do
narratives, runs, rows = {}, {}, []
for cid, risk in risk_results.items():
    try:
        out = compliance_agent.generate_compliance_narrative(cases[cid], risk)
        narratives[cid] = out
    except ComplianceError as exc:
        out = None
        print(f"{cid}: {exc}")
    run = compliance_agent.last_run          # replaced on every call, so keep a reference
    runs[cid] = run
    rows.append({
        "customer": cid,
        "typology": risk.classification,
        "accepted": out is not None,
        "steps": run.steps,
        "tools_used": " → ".join(s["action"] or "(text)" for s in run.trace),
        "words": count_words(out.narrative) if out else None,
        "warnings": len((run.report or {}).get("warnings", [])),
        "prompt_tokens": run.token_usage.get("prompt_tokens", 0),
        "completion_tokens": run.token_usage.get("completion_tokens", 0),
        "latency_s": round(run.latency_ms / 1000, 1),
    })
stage2 = pd.DataFrame(rows)
stage2

**How to read this table:** a typical good run is `get_transaction_details → lookup_regulations →
submit_narrative`. Extra `check_narrative` or repeated `submit_narrative` steps mean the model
revised a draft after the checks rejected it. Compare `prompt_tokens` with `completion_tokens`: each
step re-sends the whole conversation, so input dominates the cost.

## 4. One full ReACT trace

In [ ]:
# Phase 3 – Step 3.9: Thought → Action → Observation for the first approved case
cid = next(iter(runs), None)
if cid:
    print(f"Case {cid} ({risk_results[cid].classification})\n")
    for s in runs[cid].trace:
        print(f"── Step {s['step']}")
        if s["thought"]:
            print(textwrap.fill(f"Thought: {s['thought']}", width=110))
        print(f"Action: {s['action']}({json.dumps(s.get('arguments'))[:300]})")
        print(textwrap.fill(f"Observation: {json.dumps(s['observation'])[:600]}", width=110), "\n")

**Notice:** with `tool_choice="required"`, `gpt-4o-mini` usually sends **no `Thought:` text**, only
the tool call (a Stage 03 finding). The *actions* and *observations* are still a complete, replayable
record, and the audit log keeps them. If a `submit_narrative` was rejected, its observation lists the
exact errors the model then fixed. That is the revise-on-failure loop.

## 5. The narratives and their validation reports

In [ ]:
# Phase 3 – Step 3.9: each accepted narrative, re-checked independently, plus its citations
for cid, out in narratives.items():
    case, typology = cases[cid], risk_results[cid].classification
    report = check_narrative(out.narrative, out.regulatory_citations, case, typology)
    print(f"═══ {cid} | {typology} | {report.word_count} words | passed={report.passed} | "
          f"5 W's={all(report.five_ws.values())}")
    print(textwrap.fill(out.narrative, width=105))
    print("Citations:", "; ".join(out.regulatory_citations))
    for w in report.warnings:
        print("  ⚠", w)
    print()

**Judge them yourself:** the checks prove each narrative is short, names the subject, has real dates,
cites approved law and avoids accusatory words. They *cannot* prove the story is fair or complete.
Do the amounts and dates match the transactions? Is the "why" convincing? That judgement belongs to
the Stage 04 compliance gate (a human) and to the Stage 08 LLM-as-judge.

## 6. The audit trail

In [ ]:
# Phase 3 – Step 3.9: the ComplianceOfficer events this notebook just wrote
events = [json.loads(line) for line in AUDIT_FILE.read_text(encoding="utf-8").splitlines()]
recent = [e for e in events if e["agent_type"] == "ComplianceOfficer"][-len(runs):] if runs else []
pd.DataFrame([{k: e.get(k) for k in ["case_id", "success", "steps", "attempts", "model", "seed",
                                     "prompt_version"]}
              | {"trace_steps": len(e.get("react_trace") or []),
                 "report_passed": (e.get("validation_report") or {}).get("passed"),
                 "total_tokens": (e.get("token_usage") or {}).get("total_tokens")}
              for e in recent])

## 7. Tokens and cost per stage
Prices below are **assumptions** for `gpt-4o-mini` (USD per 1M tokens). Check the current price list
before relying on them. The audit log has exact token counts, so cost can always be recomputed.

In [ ]:
# Phase 3 – Step 3.9: estimated cost of each stage, from the audit log
PRICE_PER_M = {"prompt": 0.15, "completion": 0.60}   # assumed USD / 1M tokens — verify


def stage_cost(agent_type, n):
    evs = [e for e in events if e["agent_type"] == agent_type][-n:] if n else []
    p = sum((e.get("token_usage") or {}).get("prompt_tokens", 0) for e in evs)
    c = sum((e.get("token_usage") or {}).get("completion_tokens", 0) for e in evs)
    usd = p / 1e6 * PRICE_PER_M["prompt"] + c / 1e6 * PRICE_PER_M["completion"]
    return {"stage": agent_type, "cases": len(evs), "prompt_tokens": p, "completion_tokens": c,
            "est_usd": round(usd, 5), "usd_per_case": round(usd / max(len(evs), 1), 5)}


pd.DataFrame([stage_cost("RiskAnalyst", len(demo_ids)), stage_cost("ComplianceOfficer", len(runs))])

## 8. Takeaways
- **ReACT grounds the narrative:** amounts, dates and citations come from tool observations, not the
  model's memory, and code (not the model) decides when a narrative is acceptable.
- **The loop corrects itself:** a rejected submission returns concrete errors, and the model revises.
- **Checks ≠ quality:** passing every rule is necessary, not sufficient. Human review (Stage 04) and
  LLM-as-judge grading (Stage 08) are still needed.
- **Stage 2 costs more per case than Stage 1,** mostly re-sent input. Running it only on
  human-approved cases is the cost optimization Stage 04 will measure.